In [0]:
from pyspark.sql.functions import count, max, min, avg, sum, round
from dateutil.relativedelta import relativedelta
from datetime import date

In [0]:
two_months_ago_start = date.today().replace(day=1) - relativedelta(months=2)

In [0]:
"""
import enriched data
"""
df = spark.read.table('workspace.02_silver.yellow_trips_enriched').filter(
    f"tpep_pickup_datetime > '{two_months_ago_start}'"
)

In [0]:
"""
transform data
"""
df = df.groupBy(df.tpep_pickup_datetime.cast("date").alias("pickup_date")).agg(
        count("*").alias("total_trips"),
        round(avg("passenger_count"),1).alias("average_passengers"),
        round(avg("trip_distance"),1).alias("average_distance"),
        round(avg("fare_amount"),1).alias("average_fare_per_trip"),
        max("fare_amount").alias("max_fare"),
        min("fare_amount").alias("min_fare"),
        round(sum("total_amount"),2).alias("total_revenue")
    )                               

In [0]:
"""
write data to table
"""
df.write.saveAsTable('03_gold.daily_trip_summary', mode='append')